# House Pricing EDA

## Overview
I have looked at the dataset, and I can not give any tangible hypotheses for the feature or heirachy of features that influences the price of house the most. But based on general knowledge, I would presume the features most correlated to prices are land area, location and number of rooms. 

### Cleaning...
This is where I hunt for columns with missing rows and columns that are noise, and decide to drop them (noise col or NAN cols ) or fill them (NAN cols) 

I'll start with the NAN cols, then I'll advance to check for noise.


In [146]:
import pandas as pd

df = pd.read_csv("train.csv")

df["MSSubClass"] = df["MSSubClass"].astype(
    str
)  # Convert MSSubClass to string type because it is a categorical variable represented as numbers

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

numeric_cols = df.select_dtypes(include="number").columns.tolist()
object_cols = df.select_dtypes(include=["object", "str"]).columns.tolist()

missing

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

## What features do I Drop and Fill?
This is where I make a decision that affects my model significantly — deciding what columns stays, and whichones leave. No column will be dropped blindly without proper analysis.
I will first drop the ID column.

I will divide missing cols into `significant` and `insignificant`. 

- **Significant** in the sense that the percentage of missing rows in the col is relatively high.
- **Insignificant** in the sense that the percentage of missing rows in the col could easily be neglected

In [147]:
df.drop("Id", axis=1, inplace=True)  # Drop the 'Id' column as it is not useful for analysis

sig_missing = missing[missing > (0.1 * len(df))].sort_values(ascending=False)  # Significant missing values are those that are greater than 1% of the total number of rows in the dataset.

sig_numeric = sig_missing.index.intersection(numeric_cols)
sig_categorical = sig_missing.index.intersection(object_cols)

print(f"Significant missing values:\n{sig_missing}")

Significant missing values:
PoolQC         1453
MiscFeature    1406
Alley          1369
Fence          1179
MasVnrType      872
FireplaceQu     690
LotFrontage     259
dtype: int64


In [148]:
insig_missing = missing[missing <= (0.1 * len(df))].sort_values(ascending=False)  # insignificant missing values are those that are less than or equal to 1% of the total number of rows in the dataset.

insig_numeric = insig_missing.index.intersection(numeric_cols)
insig_categorical = insig_missing.index.intersection(object_cols)

print(f"Insignificant missing values:\n{insig_missing}")

Insignificant missing values:
GarageType      81
GarageYrBlt     81
GarageFinish    81
GarageQual      81
GarageCond      81
BsmtExposure    38
BsmtFinType2    38
BsmtQual        37
BsmtCond        37
BsmtFinType1    37
MasVnrArea       8
Electrical       1
dtype: int64


There is a noticable pattern amongst the "Garage" related columns. They all all have a sum of 81 missing cells. 

In [149]:
no_garage = df[df["GarageQual"].isnull()][["GarageType", "GarageYrBlt", "GarageFinish", "GarageQual", "GarageCond", "GarageArea"]]

print(no_garage)


     GarageType  GarageYrBlt GarageFinish GarageQual GarageCond  GarageArea
39          NaN          NaN          NaN        NaN        NaN           0
48          NaN          NaN          NaN        NaN        NaN           0
78          NaN          NaN          NaN        NaN        NaN           0
88          NaN          NaN          NaN        NaN        NaN           0
89          NaN          NaN          NaN        NaN        NaN           0
...         ...          ...          ...        ...        ...         ...
1349        NaN          NaN          NaN        NaN        NaN           0
1407        NaN          NaN          NaN        NaN        NaN           0
1449        NaN          NaN          NaN        NaN        NaN           0
1450        NaN          NaN          NaN        NaN        NaN           0
1453        NaN          NaN          NaN        NaN        NaN           0

[81 rows x 6 columns]


`GarageArea` was not part of the columns with missing cells, but I added it to show that the Garage related missing values aren't missing for no reason. Their respective garage areas (0) implies that the properties do not have a garage. It will be irresponsible of me to just fill it with with the mean or mode, because I can't make account of what does not exist; even if it's just a few rows. Instead, I will fill the alphabetic columns with "None", and the numeric ones with 0.

In [150]:
garage_cols_numeric = ["GarageYrBlt"]
garage_cols_categorical = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]

df[garage_cols_numeric] = df[garage_cols_numeric].fillna(0)
df[garage_cols_categorical] = df[garage_cols_categorical].fillna("None")

new_insig_missing = df[insig_missing.index].isnull().sum()
new_insig_missing = new_insig_missing[new_insig_missing > 0].sort_values(ascending=False)

new_insig_missing

BsmtExposure    38
BsmtFinType2    38
BsmtQual        37
BsmtCond        37
BsmtFinType1    37
MasVnrArea       8
Electrical       1
dtype: int64

`BsmtExposure`, `BsmtFinType2`, `BsmtQual`, `BsmtCond`, `BsmtFinType1` are all Basement related; similar to the Garage situation, but with a nuance. `BsmtExposure` and `BsmtFinType2` have 38 missing cells, while `BsmtQual`, `BsmtCond` and  `BsmtFinType1` have 37; 1 difference. Why? Let's investigate...


- First of all, I want to make sure that the 37's are on the same rows.

In [151]:
qual_missing = set(df[df["BsmtQual"].isnull()].index)
cond_missing = set(df[df["BsmtCond"].isnull()].index)
exposure_missing = set(df[df["BsmtExposure"].isnull()].index)
fintype1_missing = set(df[df["BsmtFinType1"].isnull()].index)
fintype2_missing = set(df[df["BsmtFinType2"].isnull()].index)

qual_missing == cond_missing == fintype1_missing

True

It returns "True", so we're good.
- Secondly, I want to check if the 38's are supersets of the 37's

In [152]:
exposure_missing.issuperset(qual_missing) & fintype2_missing.issuperset(qual_missing)

True

It returns true, so we're good again
- Thirdly, I want to confirm that the two 38's are on the same row

In [153]:
exposure_missing == fintype2_missing

False

That's a problem there. Let's look further into the respective rows of the 38's that doesn't align

In [154]:
exposure_missing.symmetric_difference(fintype2_missing)

{332, 948}

Notice how the 38's are both supersets of the 37's, but their 1 row difference is casted into two seperate rows. `{332, 948}`

In [155]:
df.loc[[332, 948], ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "TotalBsmtSF"]]

,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,TotalBsmtSF
332,Gd,TA,No,GLQ,NaN,3206
948,Gd,TA,NaN,Unf,Unf,936


I have two decisions to make here; I drop the rows, or I fill it with the mode of the coluumns.
I will go with the latter.

In [156]:
qual_missing = set(df[df["BsmtQual"].isnull()].index)
cond_missing = set(df[df["BsmtCond"].isnull()].index)
fintype1_missing = set(df[df["BsmtFinType1"].isnull()].index)
no_basement = set(df[df["TotalBsmtSF"] == 0].index)

qual_missing == cond_missing == fintype1_missing == no_basement

True

The remaining basement-related NaN values have been confirmed to correspond to houses without a basement.
But a new question just arose; are the Basement related NAN cols the only Basement columns? Lets find out

In [157]:
[col for col in df.columns if "Bsmt" in col]

['BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinSF1',
 'BsmtFinType2',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 'BsmtFullBath',
 'BsmtHalfBath']

There are other seperate Basement related columns that were not listed as NAN. Let see what they have in them.

In [158]:
bsmt_categorical = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
bsmt_numeric_complete = ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath"]

no_basement_rows = df[df[bsmt_categorical].isnull().any(axis=1)]

no_basement_rows[bsmt_numeric_complete].isnull().sum()

BsmtFinSF1      0
BsmtFinSF2      0
BsmtUnfSF       0
TotalBsmtSF     0
BsmtFullBath    0
BsmtHalfBath    0
dtype: int64

Good news, they are all numeric, and not null.
Finally, I will fill the Basement related NAN cells with `"None"`

In [159]:
bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]

df.fillna({col: "None" for col in bsmt_cols}, inplace=True)
df[bsmt_cols].isnull().sum() 

BsmtQual        0
BsmtCond        0
BsmtExposure    0
BsmtFinType1    0
BsmtFinType2    0
dtype: int64

In [160]:
new_insig_missing = df[insig_missing.index].isnull().sum()
new_insig_missing = new_insig_missing[new_insig_missing > 0].sort_values(ascending=False)

new_insig_missing

MasVnrArea    8
Electrical    1
dtype: int64

The Basement related NAN cells have been filled. Time to look at the remaining two, starting with MasVnrArea.

In [142]:
[col for col in df.columns if "MasVnr" in col]

['MasVnrType', 'MasVnrArea']

In [161]:
type_missing = set(df[df["MasVnrType"].isnull()].index)
area_missing = set(df[df["MasVnrArea"].isnull()].index)

area_missing.issubset(type_missing)

True

In [162]:
mismatch = df[df["MasVnrType"].isnull() & df["MasVnrArea"].notna()]
mismatch["MasVnrArea"].value_counts()

MasVnrArea
0.0      859
1.0        2
288.0      1
344.0      1
312.0      1
Name: count, dtype: int64